# DA4 Assignment 2 — Data Cleaning & Descriptive Analysis
**Source:** World Bank WDI, 1992–2023

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# Global style for publication-quality graphs
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.bbox': 'tight',
    'savefig.dpi': 300,
})
OUT = '../output'

## 1. Load & reshape

The WDI export is in wide format: one row per country-series, with years as columns.
We melt it to long format, then pivot so each series becomes its own column.

In [ ]:
raw = pd.read_csv('../data/raw/wdi_raw.csv', na_values=['..'])
raw = raw[raw['Country Code'].str.len().eq(3)]  # drop footer rows
print(f'{len(raw)} rows, series: {raw["Series Code"].unique()}')

In [ ]:
year_cols = [c for c in raw.columns if c[:2] in ('19', '20')]

long = raw.melt(
    id_vars=['Country Name', 'Country Code', 'Series Code'],
    value_vars=year_cols, var_name='year_raw', value_name='value'
)
long['year'] = long['year_raw'].str[:4].astype(int)
long = long.query('year <= 2023').drop(columns='year_raw')

df = long.pivot_table(
    index=['Country Name', 'Country Code', 'year'],
    columns='Series Code', values='value'
).reset_index()
df.columns.name = None

df = df.rename(columns={
    'NY.GDP.PCAP.PP.KD'   : 'gdp_pc',
    'EN.GHG.CO2.MT.CE.AR5': 'co2_total_mt',
    'SP.POP.TOTL'          : 'pop',
    'SP.URB.TOTL.IN.ZS'    : 'urban_pct',
    'EG.USE.PCAP.KG.OE'    : 'energy_pc',
})

print(df.shape)
df.head()

## 2. Construct variables

CO2 is reported in megatonnes (Mt). We divide by population to get **tonnes per person**.

Both GDP pc and CO2 pc are heavily right-skewed, so we use **log-log** specification.
This is standard in the environmental economics literature and gives a nice elasticity
interpretation: a 1% increase in GDP is associated with a β% increase in CO2.

64 country-year observations have exactly zero CO2 pc (tiny territories with negligible emissions).
These become NaN under the log transform — this is expected and acceptable.

In [ ]:
df['co2_pc'] = df['co2_total_mt'] * 1e6 / df['pop']

# Log only for positive values; zeros -> NaN
df['ln_gdp_pc'] = np.log(df['gdp_pc'].where(df['gdp_pc'] > 0))
df['ln_co2_pc'] = np.log(df['co2_pc'].where(df['co2_pc'] > 0))

n_zero = (df['co2_pc'].le(0) & df['co2_pc'].notna()).sum()
print(f'CO2 pc <= 0 (become NaN in log): {n_zero}')

## 3. Summary statistics

In [ ]:
vars_desc = ['gdp_pc', 'co2_pc', 'ln_gdp_pc', 'ln_co2_pc', 'urban_pct', 'pop']
df[vars_desc].describe().round(3).T

Both GDP and CO2 per capita show large variation across countries, as expected.
The distributions are heavily right-skewed in levels — the log transform makes them
approximately symmetric, confirming the log-log specification is appropriate.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 6))

df['gdp_pc'].dropna().plot.hist(bins=50, ax=ax[0,0], color='steelblue', edgecolor='white')
ax[0,0].set_title('GDP per capita (PPP \\$)')
ax[0,0].set_xlabel('USD')

df['ln_gdp_pc'].dropna().plot.hist(bins=50, ax=ax[0,1], color='steelblue', edgecolor='white')
ax[0,1].set_title('ln(GDP per capita)')
ax[0,1].set_xlabel('Log USD')

df['co2_pc'].dropna().plot.hist(bins=50, ax=ax[1,0], color='coral', edgecolor='white')
ax[1,0].set_title('CO$_2$ per capita (tonnes)')
ax[1,0].set_xlabel('Tonnes')

df['ln_co2_pc'].dropna().plot.hist(bins=50, ax=ax[1,1], color='coral', edgecolor='white')
ax[1,1].set_title('ln(CO$_2$ per capita)')
ax[1,1].set_xlabel('Log tonnes')

plt.tight_layout()
plt.savefig(f'{OUT}/fig_distributions.png')
plt.savefig(f'{OUT}/fig_distributions.pdf')
plt.show()

**Functional form.** The scatter plots below show ln(GDP pc) vs ln(CO2 pc) — the relationship
appears approximately linear in both cross-sections. This means the log-log specification
fits the data well: in levels, GDP and CO2 would show a curved (power-law) relationship, but
after the log transform it straightens out. We adopt **ln-ln** as our functional form throughout.
This gives the coefficient a clean interpretation as an **elasticity**: a 1% increase in GDP pc
is associated with a β% increase in CO2 pc.

The slope is stable across the two years (~0.9 in both), suggesting the cross-sectional
relationship has not changed dramatically over time.

In [ ]:
last_yr = df.dropna(subset='ln_gdp_pc')['year'].max()

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for i, yr in enumerate([2005, last_yr]):
    cs = df.query('year == @yr').dropna(subset=['ln_gdp_pc', 'ln_co2_pc'])
    ax[i].scatter(cs['ln_gdp_pc'], cs['ln_co2_pc'], alpha=0.45, s=18, color='steelblue', edgecolors='none')
    m, b = np.polyfit(cs['ln_gdp_pc'], cs['ln_co2_pc'], 1)
    xs = np.linspace(cs['ln_gdp_pc'].min(), cs['ln_gdp_pc'].max())
    ax[i].plot(xs, m*xs + b, color='firebrick', lw=2, label=f'slope = {m:.2f}')
    ax[i].set(xlabel='ln(GDP per capita)', ylabel='ln(CO$_2$ per capita)',
              title=f'Cross-section {int(yr)}  (N = {len(cs)})')
    ax[i].legend(frameon=True, fancybox=False, edgecolor='grey')

plt.tight_layout()
plt.savefig(f'{OUT}/fig_scatter.png')
plt.savefig(f'{OUT}/fig_scatter.pdf')
plt.show()

## 4. Missing value analysis

Urban % and population have zero missing values. GDP per capita is the most incomplete
variable (~10%), followed by CO2 (~6.5%). Missing GDP data drives most of the coverage
problems.

In [ ]:
n_countries = df['Country Code'].nunique()
n_years = df['year'].nunique()
print(f'{n_countries} countries x {n_years} years = {len(df)} obs\n')

miss = df[vars_desc].isna().sum()
pct  = (miss / len(df) * 100).round(1)
pd.DataFrame({'n_missing': miss, 'pct': pct})

In [ ]:
miss_yr = df.groupby('year')[['gdp_pc', 'co2_pc', 'urban_pct']].apply(
    lambda x: x.isna().mean() * 100
).round(1)

fig, ax = plt.subplots(figsize=(10, 3.5))
miss_yr.plot(marker='o', ms=4, ax=ax, linewidth=1.5)
ax.set_ylabel('Share missing (%)')
ax.set_xlabel('Year')
ax.set_title('Missing data rate by year')
ax.legend(['GDP pc', 'CO$_2$ pc', 'Urban %'], frameon=True, fancybox=False, edgecolor='grey')
plt.tight_layout()
plt.savefig(f'{OUT}/fig_missing.png')
plt.savefig(f'{OUT}/fig_missing.pdf')
plt.show()

## 5. Coverage analysis & dropping countries

We check how many years each country has all three core variables (GDP pc, CO2 pc, urban %)
non-missing. The coverage distribution is bimodal: most countries have full 32-year coverage,
but 26 countries have **zero** complete observations and 1 country (Djibouti) has only 11.

In [ ]:
core = ['gdp_pc', 'co2_pc', 'urban_pct']

cov = (df.groupby('Country Code')[core]
       .apply(lambda x: x.notna().all(axis=1).sum())
       .rename('yrs_ok').reset_index())
cov = cov.merge(df[['Country Code', 'Country Name']].drop_duplicates())
cov['pct'] = (cov['yrs_ok'] / n_years * 100).round(1)

print('Coverage distribution:')
print(cov['yrs_ok'].describe().round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
cov['yrs_ok'].plot.hist(bins=33, color='steelblue', edgecolor='white', ax=ax)
ax.set_xlabel(f'Complete years (out of {n_years})')
ax.set_ylabel('Number of countries')
ax.set_title('Data coverage per country')
plt.tight_layout()
plt.savefig(f'{OUT}/fig_coverage.png')
plt.savefig(f'{OUT}/fig_coverage.pdf')
plt.show()

In [ ]:
poor = cov.query(f'yrs_ok < {n_years // 2}').sort_values('yrs_ok')
print(f'Countries with < {n_years // 2} complete years: {len(poor)}')
print(poor[['Country Name', 'Country Code', 'yrs_ok', 'pct']].to_string(index=False))

The 27 countries with poor coverage fall into clear categories:

- **Tiny territories** with no GDP data: American Samoa, Guam, Gibraltar, Channel Islands,
  Isle of Man, Liechtenstein, Monaco, San Marino, Northern Mariana Islands, British Virgin Islands,
  New Caledonia, French Polynesia, Curacao, St. Martin, Sint Maarten
- **Politically unstable / data-poor countries**: North Korea, Cuba, Venezuela, Eritrea,
  Yemen, South Sudan, Kosovo, West Bank and Gaza
- **Recently formed states** (no data before independence): Montenegro, Serbia, Kosovo, South Sudan
- **Borderline case**: Djibouti (11 out of 32 years)

**Decision:** Drop all 27 countries. These are either not meaningful economic units for our
analysis (territories with populations under 100k) or have data gaps too large for panel methods.
Keeping them would distort the first-difference and fixed-effects models.

In [ ]:
drop_codes = poor['Country Code'].tolist()
df = df[~df['Country Code'].isin(drop_codes)]

print(f'Dropped {len(drop_codes)} countries')
print(f'Remaining: {df["Country Code"].nunique()} countries, {len(df)} obs')

## 6. Check for outliers and implausible values

Before saving, we check whether the remaining data looks plausible.
Extreme values in CO2 per capita or GDP per capita could signal data errors.

In [ ]:
# Top 10 CO2 per capita observations
print('Top 10 CO2 per capita (tonnes):')
top_co2 = df.nlargest(10, 'co2_pc')[['Country Name', 'year', 'co2_pc', 'gdp_pc', 'pop']]
print(top_co2.to_string(index=False))

In [ ]:
# Bottom 10 CO2 per capita (non-zero)
print('Bottom 10 CO2 per capita (non-zero):')
bot_co2 = df[df['co2_pc'] > 0].nsmallest(10, 'co2_pc')[['Country Name', 'year', 'co2_pc', 'gdp_pc', 'pop']]
print(bot_co2.to_string(index=False))

In [ ]:
# Countries with zero CO2 — who are they?
zero_co2 = df[df['co2_pc'] == 0][['Country Name', 'Country Code']].drop_duplicates()
print(f'Countries with any zero CO2 pc: {len(zero_co2)}')
print(zero_co2.to_string(index=False))

The top emitters (Qatar, Trinidad, Kuwait, Bahrain, UAE) are oil-rich Gulf states with small
populations and massive fossil fuel industries — these are real values, not errors.

The bottom non-zero values are very low-income African countries (Burundi, DRC, Chad, Somalia)
with minimal industrial activity — also plausible.

Countries with exactly zero CO2 are tiny Pacific/Caribbean islands where reported emissions
round to zero. These observations are lost in the log transform but remain in the dataset
for level-based models.

## 7. Final summary statistics (clean sample)

In [ ]:
print(f'Clean panel: {df["Country Code"].nunique()} countries x {n_years} years = {len(df)} obs\n')

# Build and export summary statistics table
desc = df[['gdp_pc', 'co2_pc', 'urban_pct', 'pop']].describe().T
desc = desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
desc['count'] = desc['count'].astype(int)
desc.index = ['GDP per capita (PPP \\$)', 'CO$_2$ per capita (t)', 'Urban population (\\%)', 'Population']

# Display
display(desc.round(2))

# Export LaTeX
tex = desc.round(2).to_latex(
    caption='Summary statistics --- clean panel (190 countries, 1992--2023)',
    label='tab:sumstats',
    column_format='l' + 'r' * len(desc.columns),
    escape=False,
)
with open(f'{OUT}/tab_sumstats.tex', 'w') as f:
    f.write(tex)
print(f'Saved {OUT}/tab_sumstats.tex')

## 8. Save clean dataset

In [ ]:
df.to_csv('../data/clean/wdi_clean.csv', index=False)
print(f'Saved: {df.shape}')